In [12]:
import anndata as ad
import pandas as pd

adata_path = "temp/gm12878_spliced_preprocessed_DeepCycle_with_phase_ori.h5ad"
# out_path = "temp/gm12878_spliced_preprocessed_DeepCycle_with_phase_clean_obs.h5ad"

adata = ad.read_h5ad(adata_path)

# 1. 清理 cell barcode
new_barcodes = (
    adata.obs_names.astype(str)
    .str.replace(r"^.*:", "", regex=True)  # 去掉 GM12878:
    .str.replace(r"x$", "", regex=True)    # 去掉末尾 x
)

adata.obs_names = new_barcodes
adata.obs_names_make_unique()

# 2. 只保留细胞周期标签

phase = adata.obs["cellPhase"].astype(str)


# 3. 统一细胞周期标签格式
phase = phase.replace({
    "G2/M": "G2M",
    "G2_M": "G2M",
    "G2M": "G2M",
    "S": "S",
    "G1": "G1"
})

cell_phase_df = pd.DataFrame(
    {
        "cellPhase": phase.values
    },
    index=adata.obs_names
)



In [11]:
adata.obs

,initial_size_unspliced,initial_size_spliced,initial_size,n_counts,cell_cycle_theta,DeepCycle_phase
AAACGGGCATGGTTGT,330.0,1761.0,1761.0,9719.679688,0.27,G2/M
AAACGGGGTGATAAAC,2886.0,13560.0,13560.0,11559.337891,0.02,S
AAACGGGGTTTAGCTG,4453.0,23881.0,23881.0,11188.463867,0.84,S
AAAGTAGCACGAAGCA,3575.0,25389.0,25389.0,11700.322266,0.78,S
AAAGATGAGGCTCATT,161.0,854.0,854.0,9963.859375,0.29,G2/M
...,...,...,...,...,...,...
TTTGTCACACATGGGA,1268.0,4480.0,4480.0,10722.902344,1.00,S
TTTGCGCGTTATCACG,1896.0,11888.0,11888.0,6235.700684,0.20,G2/M
TTTGTCATCCCTAATT,2393.0,11071.0,11071.0,7535.348633,0.18,G2/M
TTTGCGCCAAGGTTTC,3312.0,21962.0,21962.0,11291.263672,0.89,S


In [13]:
cell_phase_df.groupby("cellPhase").size()

cellPhase
G1      638
G2M    2259
S      5344
dtype: int64

In [14]:
origin_cell_phase_df = pd.read_csv('extra/datasets/burst/raw_data/gm12878/cellQC.tsv', sep='\t', index_col=0)
print(origin_cell_phase_df.groupby("cellPhase").size())
origin_cell_phase_df

cellPhase
G1     4679
G2M    1312
S      1256
dtype: int64


,UMI,geneNumber,mtProportion,cellPhase
AAACCTGAGACCCACC,14975,2767,0.034,G1
AAACCTGAGAGGTAGA,25084,3743,0.044,G2M
AAACCTGAGATGTCGG,13749,2614,0.037,G1
AAACCTGAGTATCGAA,20444,2799,0.041,G1
AAACCTGAGTGGCACA,10485,1796,0.066,G1
...,...,...,...,...
TTTGTCAGTACTCAAC,9659,2524,0.029,G1
TTTGTCAGTGCAACGA,20691,2769,0.031,G1
TTTGTCATCCCTAATT,14273,2143,0.028,G1
TTTGTCATCGCTGATA,26498,3634,0.034,G1


In [16]:
results = cell_phase_df.merge(origin_cell_phase_df[[]], left_index=True, right_index=True, how='inner')
print(results.groupby(["cellPhase"]).size())
results

cellPhase
G1      629
G2M    1461
S      5157
dtype: int64


,cellPhase
AAACGGGGTGATAAAC,S
AAACGGGGTTTAGCTG,S
AAAGTAGCACGAAGCA,S
AAACGGGTCCGTTGCT,S
AAACCTGAGTGGCACA,S
...,...
TTTGCGCGTAAAGTCA,S
TTTGCGCGTTATCACG,G2M
TTTGTCATCCCTAATT,G2M
TTTGCGCCAAGGTTTC,S


In [17]:
results.to_csv('extra/datasets/burst/raw_data/gm12878/cellQC_deepcycle.tsv',sep='\t',index=True)